# EPL Match Data — Exploration\n\nPurpose: build the evidence base for Feature Engineering decisions — what to keep, discard, or collapse.\n\nScope for this pass: every non-betting column gets looked at individually. Betting/odds columns (many sparsely populated across the 15+ season history) are analysed together in one combined section at the end, rather than one bookmaker at a time.\n\nNotable findings get written up separately in `docs/EDA_FINDINGS.md` — this notebook is for the analysis itself.

In [ ]:
import sys
from pathlib import Path

# Notebooks run with their own folder as the working directory, so src/
# (a sibling of notebooks/, not inside it) isn't importable by default -
# add the project root to Python's search path before importing from src.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATA_INTERIM_PATH

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(DATA_INTERIM_PATH / "matches.csv", parse_dates=["Date"])
df.shape

## 1. Target Distributions

It's well-established football knowledge that home wins are the most common outcome and draws the least common. What we're checking here is the actual split for *this specific dataset* (it can vary by league/era), plus the shape of the goal-difference distribution (the regression target).

In [ ]:
result_counts = df["FTR"].value_counts(normalize=True) * 100
result_counts = result_counts.reindex(["H", "D", "A"])

ax = result_counts.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xticklabels(["Home Win", "Draw", "Away Win"], rotation=0)
ax.set_ylabel("% of matches")
ax.set_title("Match Outcome Distribution")
for i, v in enumerate(result_counts):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.show()

result_counts

In [ ]:
df["GoalDifference"] = df["FTHG"] - df["FTAG"]

# Bins centered on each integer (edges at n-0.5) so each bar sits directly
# under its own label, spanning the full observed range (-9 to 9) so no
# extreme values get silently dropped.
lo, hi = int(df["GoalDifference"].min()), int(df["GoalDifference"].max())
bins = [x - 0.5 for x in range(lo, hi + 2)]

ax = df["GoalDifference"].plot(kind="hist", bins=bins, edgecolor="black")
ax.set_xlabel("Goal Difference (Home - Away)")
ax.set_title("Goal Difference Distribution")
plt.show()

print(f"Skewness: {df['GoalDifference'].skew():.3f}")  # 0 = symmetric
print(f"Kurtosis: {df['GoalDifference'].kurt():.3f}")  # 0 = normal-like tail weight (excess kurtosis)

df["GoalDifference"].describe()

## 2. Home Advantage Over Time

What we're checking: has the home/draw/away split stayed stable across the 16 seasons, or drifted? This is the concrete version of the "concept drift" concern raised in `docs/PROJECT_OVERVIEW.md` about including older seasons.

In [ ]:
by_season = (
    df.groupby("Season")["FTR"]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=["H", "D", "A"])
    * 100
)

ax = by_season.plot(kind="line", marker="o", figsize=(10, 5))
ax.set_ylabel("% of matches")
ax.set_title("Match Outcome % by Season")
ax.legend(["Home Win", "Draw", "Away Win"])
plt.xticks(rotation=45)
plt.show()

by_season

## 3. Match Statistics (non-betting columns)

First, split columns into "match info/stats" (per `docs/DATA_DICTIONARY.md`) vs. everything else (treated as betting/odds, analysed later as a batch).

In [ ]:
MATCH_COLUMNS = [
    "Div", "Date", "Time", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR", "HTHG", "HTAG", "HTR", "Referee",
    "HS", "AS", "HST", "AST", "HHW", "AHW", "HC", "AC",
    "HF", "AF", "HFKC", "AFKC", "HO", "AO",
    "HY", "AY", "HR", "AR", "HBP", "ABP", "Attendance",
    "Season", "GoalDifference",
]
match_columns_present = [c for c in MATCH_COLUMNS if c in df.columns]
betting_columns = [c for c in df.columns if c not in MATCH_COLUMNS]

print(f"{len(match_columns_present)} match/info/stat columns, {len(betting_columns)} betting/odds columns")
match_columns_present

In [ ]:
match_numeric_columns = df[match_columns_present].select_dtypes("number").columns.tolist()
df[match_numeric_columns].describe().T

In [ ]:
n_cols = 4
n_rows = -(-len(match_numeric_columns) // n_cols)  # ceiling division
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, match_numeric_columns):
    df[col].plot(kind="hist", bins=20, ax=ax, edgecolor="black")
    ax.set_title(col)

for ax in axes[len(match_numeric_columns):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[match_numeric_columns].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Between Match Statistics")
plt.show()

## 4. Missingness (non-betting columns)

What we're checking: which match-info/stat columns have missing data, and is it concentrated in specific seasons (e.g. older seasons lacking certain stats)?

In [ ]:
missing_pct = df[match_columns_present].isna().mean().sort_values(ascending=False) * 100
missing_pct = missing_pct[missing_pct > 0]

if missing_pct.empty:
    print("No missing values in any match/info/stat column.")
else:
    ax = missing_pct.plot(kind="barh", figsize=(8, max(2, 0.3 * len(missing_pct))))
    ax.set_xlabel("% missing")
    ax.set_title("Missingness by Column (match/info/stats only)")
    plt.tight_layout()
    plt.show()

missing_pct

In [ ]:
if not missing_pct.empty:
    missing_by_season = (
        df.groupby("Season")[missing_pct.index.tolist()]
        .apply(lambda g: g.isna().mean() * 100)
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(missing_by_season, annot=True, fmt=".0f", cmap="Reds", ax=ax)
    ax.set_title("% Missing by Column and Season")
    plt.show()

## 5. Betting/Odds Columns (combined analysis)

Rather than one bookmaker at a time: (a) how populated is each odds column overall, (b) how closely do the main providers agree with each other (redundancy check), and (c) do the odds actually predict outcomes (calibration sanity check)?

In [ ]:
coverage = (df[betting_columns].notna().mean() * 100).sort_values(ascending=False)

print(f"{(coverage >= 90).sum()} of {len(coverage)} betting columns have >=90% coverage")
print(f"{(coverage < 10).sum()} of {len(coverage)} betting columns have <10% coverage")

ax = coverage.head(30).plot(kind="barh", figsize=(8, 8))
ax.invert_yaxis()
ax.set_xlabel("% of rows populated")
ax.set_title("Top 30 Most-Populated Betting Columns")
plt.tight_layout()
plt.show()

In [ ]:
outcome_groups = {"Home": "H", "Draw": "D", "Away": "A"}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (label, suffix) in zip(axes, outcome_groups.items()):
    cols = [c for c in [f"B365{suffix}", f"Max{suffix}", f"Avg{suffix}"] if c in df.columns]
    sns.heatmap(df[cols].corr(), annot=True, fmt=".3f", cmap="coolwarm", vmin=0, vmax=1, ax=ax, cbar=False)
    ax.set_title(f"{label} Win Odds")

plt.suptitle("Correlation: Bet365 vs Market Max vs Market Average Odds")
plt.tight_layout()
plt.show()

In [ ]:
calib = df.dropna(subset=["B365H"]).copy()
calib["ImpliedHomeProb"] = 1 / calib["B365H"]
calib["ProbBucket"] = pd.cut(calib["ImpliedHomeProb"], bins=10)

bucket_stats = calib.groupby("ProbBucket", observed=True).agg(
    mean_implied_prob=("ImpliedHomeProb", "mean"),
    actual_home_win_rate=("FTR", lambda s: (s == "H").mean()),
    n_matches=("FTR", "count"),
)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(bucket_stats["mean_implied_prob"], bucket_stats["actual_home_win_rate"], marker="o", label="Actual")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
ax.set_xlabel("Bet365 implied home win probability (1/B365H)")
ax.set_ylabel("Actual home win rate")
ax.set_title("Odds Calibration Check")
ax.legend()
plt.show()

bucket_stats